# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d05_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # concrete train
    shutil.copy(glob.glob("/kaggle/input/**/d05_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # concrete test
    print("Kaggle: concrete data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: concrete data copied to ./drive/MyDrive/Regression/concrete_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from scipy.stats import genpareto
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import warnings
import io
import time
import numpy as np
import pandas as pd
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.utils.dataframe import dataframe_to_rows

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795396340.582633      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795396354.981186      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (concrete_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['C', 'mp', 'FA', 'CA', 'F', 'W_P', 'Adm']
TARGET_LABEL = 'Compressive strength (str)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred): # Corrected: Removed y_test argument as it's not defined and not used
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}

    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=N_BINS):
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_test.to_numpy()[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) & (y_test.to_numpy()[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan) # or handle empty bins as needed
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages


class visualize:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = visualize.width(int_pred)
        coverages = visualize.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        n_samples = len(y_true)
        y_true_np = y_true.to_numpy().ravel() # Ensure y_true is numpy array and flattened

        # Determine points inside and outside the interval
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval

        # Plot points inside the interval in green
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        # Plot points outside the interval in orange
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')

        # Plot all ground truth values as blue crosses
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')


        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)], linestyle='--', color='blue', label='Ideal Prediction Line') # Changed label for clarity
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=N_BINS, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()

        # Calculate interval widths for test and train sets
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])

        # Plot histograms for both train and test widths
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')


        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=N_BINS, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()

        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')

        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax


# FIX N2: the interpolating version below is NOT the NexCP weighted quantile (no +inf mass for the
# test point, linear interpolation between residuals). Renamed so it no longer overrides the
# corrected `weighted_quantile` in the helper cell; kept only for reference.
def _old_weighted_quantile_interp(values, weights, q):
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]
    cum_weights = np.cumsum(weights)
    cum_weights = cum_weights / cum_weights[-1]
    return np.interp(q, cum_weights, values)

In [13]:
# FIX N1-N9: helpers for the corrected NEXCP / online CP / COP / Extreme CP methods.
# Sources (see REFERENCES.md):
#   NexCP  - Barber, Candes, Ramdas & Tibshirani (2023), "Conformal prediction beyond exchangeability",
#            eq. (10)-(11) weighted split, eq. (12) weighted full, eq. (33) weighted jackknife+ / CV+.
#   J+aB   - Kim, Xu & Barber (2020), "Predictive inference is free with the jackknife+-after-bootstrap".
#   SF-OGD, SAOCP - Bhatnagar et al. (2023), ported line by line from the authors' code
#            github.com/salesforce/online_conformal (ogd.py, saocp.py).
#   FACI   - Gibbs & Candes (2024), ported from the same repo (faci.py); ACI - Gibbs & Candes (2021).
#   Extreme CP - Pasche, Lam & Engelke (2025), official R package ExtremeConformal (GPD_quantiles,
#            conformalize_GPD_max).
import math
from scipy.special import logsumexp

ALPHA = 0.1   # FIX M4: one miscoverage level for every method in this notebook (target coverage 90%)


# ---------------------------------------------------------------- N1: out-of-fold residuals
def _oof_fold_predictions(model_class, params, X_train, y_train, X_test, n_folds=5, seed=123):
    """Out-of-fold absolute residuals for every training row (each row scored by a model that never
    saw it), plus the fold models' predictions on X_test.
    Returns R (n,), mu_rows (n, n_test) = prediction at each test x of the fold model that left row i
    out, and fold_preds (K, n_test)."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    R = np.zeros(len(y))
    mu_rows = np.zeros((len(y), len(Xt)))
    fold_preds = []
    for tr, cal in KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(X):
        m = model_class(**params)
        m.fit(X[tr], y[tr])
        R[cal] = np.abs(y[cal] - np.asarray(m.predict(X[cal])).ravel())
        p = np.asarray(m.predict(Xt)).ravel()
        mu_rows[cal] = p[None, :]
        fold_preds.append(p)
    return R, mu_rows, np.asarray(fold_preds)


def _jab_oob(model_class, params, X_train, y_train, X_test, n_bootstrap, seed=123):
    """Jackknife+-after-bootstrap: each training row is scored only by the bootstrap models whose
    sample did NOT contain it (out-of-bag). Rows that were in every bootstrap sample are dropped."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    n = len(y)
    rng = np.random.default_rng(seed)
    in_bag = np.zeros((n_bootstrap, n), dtype=bool)
    P_tr = np.zeros((n_bootstrap, n))
    P_te = np.zeros((n_bootstrap, len(Xt)))
    for b in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        in_bag[b, idx] = True
        m = model_class(**params)
        m.fit(X[idx], y[idx])
        P_tr[b] = np.asarray(m.predict(X)).ravel()
        P_te[b] = np.asarray(m.predict(Xt)).ravel()
    oob = ~in_bag
    has = oob.any(axis=0)
    cnt = np.maximum(oob.sum(axis=0), 1)
    mu_tr = (P_tr * oob).sum(axis=0) / cnt
    R = np.abs(y - mu_tr)
    mu_rows = (oob.T.astype(float) @ P_te) / cnt[:, None]
    return R[has], mu_rows[has], P_te.mean(axis=0), has


# ---------------------------------------------------------------- N2: NexCP weighted quantile
def weighted_quantile(values, weights, q):
    """NexCP eq. (10)-(11): Q_q( sum_i w~_i delta_{R_i} + w~_{n+1} delta_{+inf} ),
    w~_i = w_i / (w_1+...+w_n+1). Returns the smallest value whose cumulative normalised weight reaches q
    (a step quantile, no interpolation), or +inf when the test point's own mass is needed.
    With all weights 1 this is the usual split-conformal ceil((n+1)(1-alpha))-th smallest residual.
    Was: normalised by w_1+...+w_n only (no test-point mass) and interpolated with np.interp."""
    v = np.asarray(values, dtype=float).ravel()
    w = np.asarray(weights, dtype=float).ravel()
    order = np.argsort(v)
    v, w = v[order], w[order]
    cum = np.cumsum(w) / (w.sum() + 1.0)
    k = np.searchsorted(cum, q - 1e-12, side="left")
    return np.inf if k >= len(v) else float(v[k])


CAP_NOTE = "interval capped: weighted quantile reached +inf (set to the largest calibration error)"


def _cap_q(q, R):
    if np.isfinite(q):
        return q, ""
    return float(np.max(R)), CAP_NOTE


def _jackknife_plus(mu_rows, R, w, alpha):
    """Weighted jackknife+ / CV+ (NexCP eq. 33): for each test point,
    lower = Q_alpha( sum w~_i delta_{mu_-i(x) - R_i} + w~_{n+1} delta_{-inf} ),
    upper = Q_{1-alpha}( sum w~_i delta_{mu_-i(x) + R_i} + w~_{n+1} delta_{+inf} ).
    Infinite bounds are capped at mu +/- max(R) and counted."""
    lo, hi, n_cap = [], [], 0
    for j in range(mu_rows.shape[1]):
        mu = mu_rows[:, j]
        l = -weighted_quantile(-(mu - R), w, 1 - alpha)
        h = weighted_quantile(mu + R, w, 1 - alpha)
        if not (np.isfinite(l) and np.isfinite(h)):
            n_cap += 1
            l = l if np.isfinite(l) else float(np.mean(mu) - np.max(R))
            h = h if np.isfinite(h) else float(np.mean(mu) + np.max(R))
        lo.append(l)
        hi.append(h)
    return np.asarray(lo), np.asarray(hi), n_cap


def _cap_count_note(n_cap, n):
    return "" if n_cap == 0 else f"{n_cap} of {n} intervals capped: weighted quantile reached +inf"


# ---------------------------------------------------------------- N3: NexCP full conformal (eq. 12)
def _nexcp_full_point(model_class, params, X, y, x_new, grid, wt, alpha):
    """FIX B15: edge scan for one test point. Only the smallest and the largest kept y set the interval,
    so scan up from the bottom of the grid to the first kept y, then down from the top to the first kept y.
    Same keep rule and same [lo, hi] as checking every grid value; points in the middle are skipped."""
    X_aug = np.vstack([X, x_new])

    def kept(yc):
        y_aug = np.append(y, yc)
        m = model_class(**params)
        m.fit(X_aug, y_aug)
        r = np.abs(y_aug - np.asarray(m.predict(X_aug)).ravel())
        return wt[r[:-1] < r[-1]].sum() < 1 - alpha

    lo_i = next((k for k in range(len(grid)) if kept(grid[k])), None)
    if lo_i is None:
        return None, None
    hi_i = next((k for k in range(len(grid) - 1, lo_i, -1) if kept(grid[k])), lo_i)
    return lo_i, hi_i


def _nexcp_full(model_class, params, X_train, y_train, X_test, yhat_test, w, alpha, half_range,
                grid_size=50, label=""):
    """For each test x and each candidate y on a grid of `grid_size` values (prediction +/- half_range):
    refit on train + (x, y), compute all residuals R^y, and keep y if the test residual is not above
    the weighted (1-alpha) quantile of { R^y_i (weight w~_i), R^y_{n+1} (weight w~_{n+1}) }.
    The interval is [smallest kept y, largest kept y].
    FIX B15 (speed only, same result): edge scan per test point, test points split over 2 workers."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    wt = np.asarray(w, dtype=float) / (np.sum(w) + 1.0)
    grids = [np.linspace(yhat_test[j] - half_range, yhat_test[j] + half_range, grid_size) for j in range(len(Xt))]
    t0 = time.time()
    args = [(model_class, params, X, y, Xt[j:j + 1], grids[j], wt, alpha) for j in range(len(Xt))]
    # 2 workers help most models, but a model that already uses every core (e.g. LightGBM) can get much
    # slower when two copies run at once. So: time a few points one at a time, a few with 2 workers, and
    # use the faster way for the rest. The result per point is the same either way.
    res = [None] * len(Xt)

    def run_seq(idx):
        for k in idx:
            res[k] = _nexcp_full_point(*args[k])

    def run_par(idx):
        from joblib import Parallel, delayed, parallel_config
        with parallel_config(backend="loky", inner_max_num_threads=2):
            out = Parallel(n_jobs=2, batch_size=1, verbose=0)(delayed(_nexcp_full_point)(*args[k]) for k in idx)
        for k, r in zip(idx, out):
            res[k] = r

    n_seq, n_par = 3, 6
    how = "1 worker"
    if len(Xt) >= 2 * (n_seq + n_par):
        t1 = time.time(); run_seq(range(n_seq)); t_seq = (time.time() - t1) / n_seq
        try:
            t1 = time.time(); run_par(range(n_seq, n_seq + n_par)); t_par = (time.time() - t1) / n_par
        except Exception as e:  # e.g. a model class that cannot be sent to a worker process
            print(f"  NexCP-Full {label}: 2 workers failed ({type(e).__name__}: {e}); using 1")
            run_seq(range(n_seq, n_seq + n_par)); t_par = float("inf")
        rest = range(n_seq + n_par, len(Xt))
        print(f"  NexCP-Full {label}: {t_seq:.1f} s/point with 1 worker, {t_par:.1f} s/point with 2")
        if t_par < 0.85 * t_seq:
            run_par(rest); how = "2 workers"
        else:
            run_seq(rest)
    else:
        run_seq(range(len(Xt)))
    print(f"  NexCP-Full {label}: {len(Xt)} test points, {(time.time() - t0) / 60:.1f} min ({how})")
    lo, hi, n_edge, n_empty = [], [], 0, 0
    for j, (lo_i, hi_i) in enumerate(res):
        if lo_i is None:
            lo.append(yhat_test[j])
            hi.append(yhat_test[j])
            n_empty += 1
        else:
            lo.append(grids[j][lo_i])
            hi.append(grids[j][hi_i])
            n_edge += int(lo_i == 0 or hi_i == grid_size - 1)
    notes = []
    if n_edge:
        notes.append(f"{n_edge} of {len(Xt)} intervals reach the edge of the y-grid (true interval may be wider)")
    if n_empty:
        notes.append(f"{n_empty} empty sets (set to the point prediction)")
    return np.asarray(lo), np.asarray(hi), "; ".join(notes)


# ---------------------------------------------------------------- N4/N6: SF-OGD and SAOCP (salesforce/online_conformal)
def _pinball(y, yhat, q):
    return np.maximum(q * (y - yhat), (1 - q) * (yhat - y))


def _pinball_grad(y, yhat, q):
    return -q * (y > yhat) + (1 - q) * (y < yhat)


def _sf_ogd(calib, stream, alpha):
    """Scale-free OGD on the pinball loss (ogd.py ScaleFreeOGD): warm-started on the calibration
    residuals, then for each test point the margin is OUTPUT first and updated after its error is seen."""
    coverage = 1 - alpha
    scale = float(np.max(np.abs(calib))) * np.sqrt(3) if len(calib) else 1.0
    state = {"delta": 0.0, "gn": 0.0}

    def update(s):
        g = _pinball_grad(abs(s), state["delta"], coverage)
        state["gn"] += g ** 2
        if state["gn"] != 0:
            state["delta"] = max(0.0, state["delta"] - scale / np.sqrt(3 * state["gn"]) * g)

    for s in calib:
        update(s)
    out = []
    for s in stream:
        out.append(state["delta"])
        update(s)
    return np.asarray(out)


class _SAOCPExpert:
    """saocp.py _OGD: scale-free OGD expert started at time t with lifetime g * 2^u."""

    def __init__(self, t, scale, alpha, yhat_0, g=8):
        self.scale = scale
        self.base_lr = scale / np.sqrt(3)
        self.alpha = alpha
        self.yhat = yhat_0
        self.grad_norm = 0
        u = 0
        while t % 2 == 0:
            t /= 2
            u += 1
        self.lifetime = g * 2 ** u
        self.z = 0
        self.wz = 0
        self.s_t = 0

    @property
    def expired(self):
        return self.s_t > self.lifetime

    def loss(self, y):
        return _pinball(y, self.yhat, 1 - self.alpha)

    @property
    def w(self):
        return 0 if self.s_t == 0 else self.z / self.s_t * (1 + self.wz)

    def update(self, y, meta_loss):
        w = self.w
        g = np.clip((meta_loss - self.loss(y)) / self.scale / max(self.alpha, 1 - self.alpha), -1 * (w > 0), 1)
        self.z += g
        self.wz += g * w
        self.s_t += 1
        grad = _pinball_grad(y, self.yhat, 1 - self.alpha)
        self.grad_norm += grad ** 2
        if self.grad_norm != 0:
            self.yhat = max(0, self.yhat - self.base_lr / np.sqrt(self.grad_norm) * grad)


def _saocp(calib, stream, alpha, lifetime=8):
    """saocp.py SAOCP (coin-betting combination of SF-OGD experts started at every step)."""
    coverage = 1 - alpha
    scale = float(np.max(np.abs(calib))) * np.sqrt(3) if len(calib) else 1.0
    experts = {}
    state = {"t": 1}

    def get_p():
        if not experts:
            return {}
        prior = {t: 1 / (t ** 2 * (1 + np.floor(np.log2(t)))) for t in experts}
        z = sum(prior.values())
        prior = {t: v / z for t, v in prior.items()}
        p = {t: prior[t] * max(0, e.w) for t, e in experts.items()}
        sp = sum(p.values())
        return {t: v / sp for t, v in p.items()} if sp > 0 else prior

    def predict():
        p = get_p()
        return sum(p[t] * e.yhat for t, e in experts.items())

    def update(s):
        s = abs(s)
        s_hat = predict()
        for t in [k for k, e in experts.items() if e.expired]:
            experts.pop(t)
        experts[state["t"]] = _SAOCPExpert(state["t"], scale, 1 - coverage, yhat_0=s_hat, g=lifetime)
        meta_loss = _pinball(s, predict(), coverage)
        for e in experts.values():
            e.update(s, meta_loss)
        state["t"] += 1

    for s in calib:
        update(s)
    out = []
    for s in stream:
        out.append(predict())
        update(s)
    return np.asarray(out)


# ---------------------------------------------------------------- N5: FACI (faci.py) and ACI
def _faci(calib, stream, alpha):
    """Fully Adaptive Conformal Inference: 8 ACI experts with step sizes 0.001 * 2^k, combined with
    exponential weights on the pinball loss (Gibbs & Candes 2024, Algorithm 1; defaults of faci.py)."""
    coverage = 1 - alpha
    gammas = np.asarray([0.001 * 2 ** k for k in range(8)])
    k = len(gammas)
    I = 100
    sigma = 1 / (2 * I)
    a = 1 - coverage
    denom = ((1 - a) ** 2 * a ** 3 + a ** 2 * (1 - a) ** 3) / 3
    eta = np.sqrt(3 / I) * np.sqrt((np.log(I * k) + 2) / denom)
    alphas = np.full(k, a)
    log_w = np.zeros(k)
    residuals = list(np.abs(calib))
    out = []
    for s in np.abs(stream):
        w = np.exp(log_w - logsumexp(log_w))
        a_t = float(np.dot(w, alphas))
        out.append(float(np.quantile(residuals, np.clip(1 - a_t, 0, 1), method="inverted_cdf")))
        if len(residuals) > math.floor(1 / (1 - coverage)):
            beta = np.mean(np.asarray(residuals) >= s)
            losses = _pinball(beta, alphas, 1 - coverage)
            wbar = log_w - eta * losses
            log_w = logsumexp([wbar, np.full(k, logsumexp(wbar))], b=[[1 - sigma], [sigma / k]], axis=0)
            log_w = log_w - logsumexp(log_w)
            err = alphas > beta
            alphas = np.clip(alphas + gammas * ((1 - coverage) - err), 0, 1)
        residuals.append(s)
    return np.asarray(out)


def _aci(calib, stream, alpha, gamma=0.005):
    """Adaptive Conformal Inference (Gibbs & Candes 2021): alpha_{t+1} = alpha_t + gamma (alpha - err_t).
    If alpha_t <= 0 the ACI set is the whole line; it is capped at the largest error seen and counted."""
    a_t = alpha
    residuals = list(np.abs(calib))
    out, n_cap = [], 0
    for s in np.abs(stream):
        level = 1 - a_t
        if level >= 1:
            q = float(np.max(residuals))
            n_cap += 1
        elif level <= 0:
            q = 0.0
        else:
            q = float(np.quantile(residuals, level, method="inverted_cdf"))
        out.append(q)
        a_t = a_t + gamma * (alpha - float(s > q))
        residuals.append(s)
    return np.asarray(out), n_cap


# ---------------------------------------------------------------- N9: Extreme CP (ExtremeConformal GPD_max)
def _gpd_quantile(p, p0, u, sigma, xi):
    """ExtremeConformal GPD_quantiles: q = u + (sigma/xi) [((1-p)/(1-p0))^(-xi) - 1]  (xi = 0: log form)."""
    if abs(xi) < 1e-12:
        return u + sigma * np.log((1 - p0) / (1 - p))
    return u + (sigma / xi) * (((1 - p) / (1 - p0)) ** (-xi) - 1)


def _conformalize_gpd_max(s_cal, coverage, threshold_lvls=None, min_obs_gpd=10):
    """ExtremeConformal conformalize_GPD_max: for each threshold level p0 (default 0.80..0.99, capped so
    at least `min_obs_gpd` scores lie above it) fit a GPD to the exceedances over u = quantile(p0),
    compute the extrapolated `coverage` quantile, and return the maximum over thresholds."""
    s_cal = np.asarray(s_cal, dtype=float).ravel()
    n = len(s_cal)
    lv = np.round(np.arange(0.80, 0.9901, 0.01), 2) if threshold_lvls is None else np.asarray(threshold_lvls, float)
    if min_obs_gpd > 0:
        lv = np.minimum(lv, 1 - min_obs_gpd / n)
    lv = np.unique(lv)
    rows = []
    for p0 in lv:
        u = float(np.quantile(s_cal, p0))
        exc = s_cal[s_cal > u] - u
        try:
            xi, _, sigma = genpareto.fit(exc, floc=0)
            dq = float(_gpd_quantile(coverage, p0, u, sigma, xi))
        except Exception:
            xi = sigma = dq = np.nan
        rows.append({"threshold_level": float(p0), "threshold_u": u, "n_exceedances": int(len(exc)),
                     "sigma": float(sigma), "xi": float(xi), "q_threshold": dq})
    table = pd.DataFrame(rows)
    finite = table["q_threshold"].replace([np.inf, -np.inf], np.nan).dropna()
    return (float(finite.max()) if len(finite) else np.nan), table


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_NEXCP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
):

    warnings.filterwarnings("ignore")

    SEED = 123
    np.random.seed(SEED)

    strategies_names = {
        "NexCP-Split": "NexCP Split",
        "NexCP-Full": "NexCP Full",
        "NexCP-J+aB": "NexCP Jackknife+ aB",
        "NexCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, SEED)

    # NexCP - Split, eq. (11)
    q, notes["NexCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NexCP-Split"] = yhat_test
    int_pred["NexCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NexCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NexCP-Full"] = yhat_test
    int_pred["NexCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NexCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-J+aB"] = bag_mean
    int_pred["NexCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NexCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NexCP-CV+"] = np.c_[lo, hi]

    # =========================================================
    # Metrics
    # =========================================================
    def coverage(y, l, u):
        return np.mean((y >= l) & (y <= u))

    def width(l, u):
        return np.mean(u - l)

    def cwc(l, u, y, alpha):
        cov = coverage(y, l, u)
        w = width(l, u)
        penalty = np.exp(-50 * (cov - (1 - alpha)))
        return w * (1 + penalty if cov < (1 - alpha) else 1)

    y_true = y_test.to_numpy()

    summary_rows = []

    print("\n===== NEXCP RESULTS =====")

    for s in strategies_names:
        cov = coverage(y_true, int_pred[s][:, 0], int_pred[s][:, 1])
        wid = width(int_pred[s][:, 0], int_pred[s][:, 1])
        rmse = np.sqrt(mean_squared_error(y_true, y_pred[s]))
        cwc_val = cwc(int_pred[s][:, 0], int_pred[s][:, 1], y_true, alpha)

        summary_rows.append([
            strategies_names[s],
            cov,
            wid,
            rmse,
            cwc_val,
        ])

        print(f"{strategies_names[s]} - Coverage: {cov*100:.3f}% | Width: {wid:.3f}")

    summary_df = pd.DataFrame(
        summary_rows,
        columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"],
    )
    summary_df["Note"] = [notes.get(s, "") for s in strategies_names]  # FIX N10: caps / limits per method

    # =========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # =========================================================
    idx = np.arange(len(y_test))
    plot_buffers = {}

    for s in strategies_names:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, y_test, color="blue", s=10, alpha=0.7, label="True")
        ax.fill_between(
            idx,
            int_pred[s][:, 0],
            int_pred[s][:, 1],
            color="gray",
            alpha=0.5,
            label="Prediction interval",
        )
        ax.plot(idx, y_pred[s], color="red", linewidth=1, label="Predicted mean")

        ax.set_title(strategies_names[s])
        ax.set_xlabel("Sample Number")
        ax.set_ylabel(f"{TARGET_LABEL}")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # =========================================================
    # Excel Export
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary sheet ----
        sheet_name = "NEXCP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name, 0)
        for r_idx, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c_idx, val in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=val)

        # ---- Plot sheets ----
        for s, buf in plot_buffers.items():
            img = Image(buf)

            sheet_name = f"NEXCP_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1

            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values sheet ----
        results_dict = {}
        for s in strategies_names:
            results_dict[f"{s}_true"] = y_true
            results_dict[f"{s}_pred"] = y_pred[s]
            results_dict[f"{s}_lower"] = int_pred[s][:, 0]
            results_dict[f"{s}_upper"] = int_pred[s][:, 1]

        df_vals = pd.DataFrame(results_dict)

        sheet_name = "NEXCP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r_idx, row in enumerate(dataframe_to_rows(df_vals, index=False, header=True), 1):
            for c_idx, val in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_online_super(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30,
):

    warnings.filterwarnings("ignore")

    SEED = 123
    np.random.seed(SEED)

    STRATEGY_NAMES = {
        "Online-Split": "Online Split",
        "Online-CV+": "Online CV+",
        "Online-J+aB": "Online Jackknife+ aB",
        "SAOCP": "SAOCP",
        "SF-OGD": "SF-OGD",
        "FACI": "FACI",
        "ACI": "ACI",   # FIX N5: added (Gibbs & Candes 2021, gamma = 0.005)
        "MFCS-Split": "MFCS Split",
        "MFCS-Full": "MFCS Full",
    }

    ALL_STRATEGIES = list(STRATEGY_NAMES.keys())
    if strategies == "all":
        strategies = ALL_STRATEGIES

    # =========================================================
    # FIX N1, N4-N6: corrected online methods. Every method starts from OUT-OF-FOLD training residuals;
    # test points arrive one at a time and each interval is OUTPUT before that point's error is used.
    # Was: in-sample training residuals; SF-OGD with flipped gradient sign and look-ahead; "FACI" and
    # "SAOCP" were window heuristics (FACI also looked ahead); J+aB scored models on their own samples.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    y_te = np.asarray(y_test).ravel()
    n_train, n_test = len(y_tr), len(y_te)

    base_model = model_class(**model_params)
    base_model.fit(X_tr, y_tr)
    yhat_te = np.asarray(base_model.predict(X_te)).ravel()
    res_te = np.abs(y_te - yhat_te)
    R_oof, mu_te_folds, _ = _oof_fold_predictions(model_class, model_params, X_tr, y_tr, X_te, 5, SEED)

    y_pred, int_pred, notes = {}, {}, {}

    def _band(name, centre, q):
        y_pred[name] = np.asarray(centre)
        int_pred[name] = np.c_[centre - q, centre + q]

    if "Online-Split" in strategies:
        q, notes["Online-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("Online-Split", yhat_te, q)

    if "Online-CV+" in strategies:
        # CV+ (Barber et al. 2021); past test points join the calibration set once their error is known
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_te_folds[:, j], np.full(j, yhat_te[j])]
            R = np.r_[R_oof, res_te[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-CV+"] = yhat_te
        int_pred["Online-CV+"] = np.c_[lo, hi]
        notes["Online-CV+"] = _cap_count_note(n_cap, n_test)

    if "Online-J+aB" in strategies:
        # J+aB (Kim, Xu & Barber 2020): out-of-bag residuals; past test points join as above
        R_oob, mu_oob, bag_mean, has = _jab_oob(model_class, model_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
        res_bag = np.abs(y_te - bag_mean)
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_oob[:, j], np.full(j, bag_mean[j])]
            R = np.r_[R_oob, res_bag[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-J+aB"] = bag_mean
        int_pred["Online-J+aB"] = np.c_[lo, hi]
        notes["Online-J+aB"] = _cap_count_note(n_cap, n_test)

    if "SAOCP" in strategies:
        _band("SAOCP", yhat_te, _saocp(R_oof, res_te, alpha))

    if "SF-OGD" in strategies:
        _band("SF-OGD", yhat_te, _sf_ogd(R_oof, res_te, alpha))

    if "FACI" in strategies:
        _band("FACI", yhat_te, _faci(R_oof, res_te, alpha))

    if "ACI" in strategies:
        q_seq, n_cap = _aci(R_oof, res_te, alpha, gamma=0.005)
        _band("ACI", yhat_te, q_seq)
        notes["ACI"] = "" if n_cap == 0 else f"{n_cap} of {n_test} steps had alpha_t <= 0 (infinite ACI set, capped at the largest error)"

    if "MFCS-Split" in strategies:
        # kept as written by the user (no published source found); same computation as Online-Split
        q, notes["MFCS-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("MFCS-Split", yhat_te, q)

    if "MFCS-Full" in strategies:
        # refit on all points seen so far; margin from that refit's OUT-OF-FOLD residuals (FIX N1)
        lo, hi, preds, n_cap = [], [], [], 0
        for j in range(n_test):
            X_h = np.vstack([X_tr, X_te[:j]])
            y_h = np.r_[y_tr, y_te[:j]]
            m = model_class(**model_params)
            m.fit(X_h, y_h)
            yhat = float(np.asarray(m.predict(X_te[j:j + 1])).ravel()[0])
            R_h, _, _ = _oof_fold_predictions(model_class, model_params, X_h, y_h, X_te[j:j + 1], 5, SEED)
            q = weighted_quantile(R_h, np.ones(len(R_h)), 1 - alpha)
            if not np.isfinite(q):
                q, n_cap = float(np.max(R_h)), n_cap + 1
            preds.append(yhat); lo.append(yhat - q); hi.append(yhat + q)
        y_pred["MFCS-Full"] = np.asarray(preds)
        int_pred["MFCS-Full"] = np.c_[lo, hi]
        notes["MFCS-Full"] = _cap_count_note(n_cap, n_test)

    # =========================================================
    # METRICS
    # =========================================================
    y_true = y_test.to_numpy().ravel()

    rows = []
    for s in y_pred:
        ymin, ymax = int_pred[s].T
        cov = np.mean((y_true >= ymin) & (y_true <= ymax))
        wid = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred[s]))
        cwc = wid * (1 + (cov < (1 - alpha)) * (1 - cov))

        rows.append([STRATEGY_NAMES[s], cov, wid, rmse, cwc])

    summary_df = pd.DataFrame(
        rows, columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"]
    )
    summary_df["Note"] = [notes.get(s, "") for s in y_pred]  # FIX N10: caps / limits per method

    print("\n===== ONLINE CP RESULTS =====")
    print(summary_df)

    # =========================================================
    # PLOTTING (SAVE ONLY)
    # =========================================================
    plot_buffers = {}
    idx = np.arange(len(y_test))

    for s in y_pred:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, y_true, color="blue", s=10, alpha=0.7, label="True")
        ax.fill_between(idx,
                        int_pred[s][:, 0],
                        int_pred[s][:, 1],
                        color="gray",
                        alpha=0.5,
                        label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", linewidth=1, label="Predicted mean")

        ax.set_title(f"{STRATEGY_NAMES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel(f"{TARGET_LABEL}")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # =========================================================
    # EXCEL EXPORT (APPEND SAFE)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "ONLINECP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plots ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"ONLINECP_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values ----
        values = {}
        for s in y_pred:
            values[f"{s}_true"] = y_true
            values[f"{s}_pred"] = y_pred[s]
            values[f"{s}_lower"] = int_pred[s][:, 0]
            values[f"{s}_upper"] = int_pred[s][:, 1]

        df_vals = pd.DataFrame(values)

        sheet_name = "ONLINECP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(df_vals, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_COP(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    lr=0.05,          # Learning rate for COP
    T_burnin=COP_T_BURNIN,      # Window size for distribution estimation
    scale=0.5,        # Scale factor for the integrator
    random_state=123,
):
    """
    Conformal Optimistic Prediction (COP)
    ✔ Implements the Distribution-Informed Gradient logic from the repository.
    ✔ Matches the "Excel-safe" and plotting structure of your previous functions.
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # 1. Train Base Model & Generate Test Predictions
    # =========================================================
    # COP is an online calibration method, but we usually start with a pre-trained model
    # or an online model. Here we fit on Train and calibrate on Test stream.
    model = model_class(**model_params)
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred_test = model.predict(X_test).ravel()
    y_true_test = y_test.to_numpy().ravel()

    # Calculate non-conformity scores (Absolute Residuals)
    # This assumes symmetric intervals.
    scores = np.abs(y_true_test - y_pred_test)
    T_test = len(scores)

    # =========================================================
    # 2. COP Initialization
    # =========================================================
    # Initialize quantiles based on training data residuals if possible
    y_pred_train = model.predict(X_train).ravel()
    scores_train = np.abs(y_train.to_numpy().ravel() - y_pred_train)
    # FIX N1: start from OUT-OF-FOLD training residuals (was: in-sample residuals, too small)
    R_oof_cop, _, _ = _oof_fold_predictions(model_class, model_params, np.asarray(X_train),
                                            np.asarray(y_train).ravel(), np.asarray(X_test)[:1], 5, 123)
    init_q = np.quantile(R_oof_cop, 1 - alpha)

    # Arrays to store history
    qs = np.zeros(T_test)          # Final adaptive quantile
    qts = np.zeros(T_test)         # Base quantile term
    integrators = np.zeros(T_test) # Optimistic integrator term
    covereds = np.zeros(T_test)    # Coverage boolean

    # Set initial values
    qs[0] = init_q
    qts[0] = init_q

    # =========================================================
    # 3. Online COP Loop
    # =========================================================
    # Logic adapted from `core/methods.py` function `COP`

    for t in range(T_test - 1):
        # 1. Observe coverage at current step t
        covereds[t] = 1 if qs[t] >= scores[t] else 0

        # FIX N8: step size proportional to the score scale, as in the official COP code
        # (lr x (max - min) of the last T_burnin scores). Was: a fixed lr = 0.05 in target units.
        _hist = np.r_[R_oof_cop, scores[:t + 1]][-T_burnin:]
        lr_t = lr * (np.max(_hist) - np.min(_hist))

        # 2. Calculate Standard Gradient
        # If covered, we decrease q (grad is positive). If not, increase q.
        grad = alpha if covereds[t] else -(1 - alpha)

        # 3. Calculate Distribution-Informed Gradient (The "Optimistic" part)
        if t < T_burnin:
            grad_i = 0
        else:
            # Look at the window of recent scores
            window_s = scores[t - T_burnin : t]

            # Calculate ECDF at the predicted next quantile step
            # logic: np.mean(window_s <= qts[t] - lr * grad) - (1 - alpha)
            current_target = qts[t] - lr_t * grad   # FIX N8
            grad_i = np.mean(window_s <= current_target) - (1 - alpha)

        # 4. Update Terms
        integrator = -scale * grad_i

        # Update base quantile (Gradient Descent)
        qts[t + 1] = qts[t] - lr_t * grad   # FIX N8

        # Update integrator state
        integrators[t + 1] = lr_t * integrator   # FIX N8

        # Combine for final quantile
        qs[t + 1] = qts[t + 1] + integrators[t + 1]

        # Safety: Ensure quantile implies non-negative width
        qs[t + 1] = max(0, qs[t + 1])

    # Handle the final step coverage for metrics
    covereds[-1] = 1 if qs[-1] >= scores[-1] else 0

    # =========================================================
    # 4. Construct Prediction Intervals
    # =========================================================
    ymin = y_pred_test - qs
    ymax = y_pred_test + qs

    # =========================================================
    # 5. Metrics
    # =========================================================
    rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
    coverage = np.mean(covereds)
    avg_width = np.mean(ymax - ymin)

    print(f"\n===== CONFORMAL OPTIMISTIC PREDICTION ({model_name}) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Avg Width: {avg_width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"Final q  : {qs[-1]:.3f}")

    # =========================================================
    # 6. Summary Tables
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "COP",
        "Coverage": coverage,
        "Width": avg_width,
        "RMSE": rmse,
        "Alpha": alpha,
        "LR": lr,
        "Scale": scale
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true_test)),
        "y_true": y_true_test,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": ymax - ymin,
        "is_covered": covereds.astype(int),
        "score_residual": scores,
        "adaptive_q": qs
    })

    # =========================================================
    # 7. Plotting
    # =========================================================
    idx = np.arange(len(y_true_test))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, y_true_test, color="blue", s=10, alpha=0.7, label="True")
    ax.fill_between(idx, ymin, ymax, color="gray", alpha=0.5, label="COP Interval")
    ax.plot(idx, y_pred_test, color="red", linewidth=1, label="Predicted mean")

    ax.set_title(f"Conformal Optimistic Prediction (COP) - {model_name}")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel(f"{TARGET_LABEL}")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # 8. Excel Export
    # =========================================================
    if excel_file_path:
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            # Create a new workbook if it doesn't exist
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active) # Remove default sheet

        # ---- Summary ----
        sheet_name = "COP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "COP_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "COP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_EXTREME(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    calib_size=0.2,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30,
    random_state=123,
):
    """
    Extreme Conformal Prediction (upper-tail, single-sided)
    ✔ Same structure as NexCP & OnlineCP
    ✔ No return values
    ✔ Excel-safe
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # Train / calibration split
    # =========================================================
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train,
        y_train,
        test_size=calib_size,
        random_state=random_state,
        shuffle=True,
    )

    # =========================================================
    # Train model
    # =========================================================
    model = model_class(**model_params)
    model.fit(X_fit, y_fit)

    y_pred_test = model.predict(X_test).ravel()
    y_pred_calib = model.predict(X_calib).ravel()

    # =========================================================
    # Nonconformity scores (upper tail only)
    # =========================================================
    # FIX N9: Extreme CP as in the official ExtremeConformal code (conformalize_GPD_max):
    # signed scores y - y_hat on ALL calibration points, thresholds 0.80..0.99 (capped so >= 10 scores
    # exceed them), official GPD quantile formula, maximum over thresholds; ONE-SIDED upper bound.
    # Was: only positive scores kept, (1/alpha)^xi in the formula (missing the threshold share), a hard
    # error with < 30 exceedances, and a two-sided band reported for a one-sided method.
    scores = np.asarray(y_calib).ravel() - y_pred_calib
    q_extreme, gpd_table = _conformalize_gpd_max(scores, 1 - alpha, min_obs_gpd=10)
    extreme_note = ""
    if not np.isfinite(q_extreme):
        q_extreme = weighted_quantile(scores, np.ones(len(scores)), 1 - alpha)
        extreme_note = "GPD fit failed at every threshold; empirical conformal quantile used"
    best_row = gpd_table.loc[gpd_table["q_threshold"].idxmax()] if gpd_table["q_threshold"].notna().any() else None
    u = float(best_row["threshold_u"]) if best_row is not None else np.nan
    n_exc_used = int(best_row["n_exceedances"]) if best_row is not None else 0
    print(gpd_table.to_string(index=False))

    # one-sided interval (-inf, y_hat + q]
    y_pred_test = np.asarray(y_pred_test).ravel()
    ymin = np.array([None] * len(y_pred_test), dtype=object)   # no lower bound (empty cell in Excel)
    ymax = y_pred_test + q_extreme

    # =========================================================
    # Metrics
    # =========================================================
    y_true = y_test.to_numpy().ravel()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred_test))
    coverage = np.mean(y_true <= ymax)
    width = q_extreme  # meaningful one-sided width

    print(f"\n===== EXTREME CONFORMAL PREDICTION ({model_name}) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Width    : {width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"q_extreme: {q_extreme:.3f}")
    print(f"u        : {u:.3f}")
    print(f"#Exceed. : {n_exc_used}")
    if extreme_note:
        print(f"Note     : {extreme_note}")

    # =========================================================
    # Summary table
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "EXTREME CP",
        "Coverage": coverage,
        "Width": width,
        "RMSE": rmse,
        "q_extreme": q_extreme,
        "threshold_u": u,
        "n_exceedances": n_exc_used,
        "Note": extreme_note,   # FIX N10
    }])

    # =========================================================
    # Values table
    # =========================================================
    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": np.full(len(y_pred_test), q_extreme),   # FIX N9: one-sided width = q
        "is_covered": (y_true <= ymax).astype(int),
        "residual": y_true - y_pred_test,
    })

    # =========================================================
    # Plot (SAVE ONLY)
    # =========================================================
    idx = np.arange(len(y_true))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(idx, y_true, s=10, alpha=0.7, label="True")
    ax.plot(idx, ymax, ls="--", lw=1, label="Upper bound (one-sided)")   # FIX N9
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Prediction")

    ax.set_title(f"Extreme Conformal Prediction ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel(f"{TARGET_LABEL}")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # Excel export (append-safe, SAME FILE)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "EXTREMECP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "EXTREMECP_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "EXTREMECP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # FIX N9: GPD fit at every threshold level (the reported q is the maximum)
        sheet_name = "EXTREMECP_GPD_Thresholds"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        _tbl = gpd_table.astype(object).where(gpd_table.notna(), None)
        for r, row in enumerate(dataframe_to_rows(_tbl, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


In [18]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path



import os
try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("[Info] Install tqdm for progress bars: pip install tqdm")


# ═══════════════════════════════════════════════════════════
# AlphaNet Architecture  (FIXED)
# ═══════════════════════════════════════════════════════════
class AlphaNet(nn.Module):
    """
    Maps a feature vector (sum of LOO scores) → learned α̃.
    Output is clamped to (alpha_min, 1) so that the conformal
    denominator α̃·(n+1)−1 is always positive.
    """

    def __init__(self, input_dim, alpha_min=0.05):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.alpha_min = alpha_min  # lower bound for valid α̃
        self._initialize_near_one()

    def _initialize_near_one(self):
        """Bias output toward α̃ ≈ 1 at init (sigmoid(5) ≈ 0.993)."""
        with torch.no_grad():
            self.fc2.weight.data.normal_(0, 0.01)
            self.fc2.bias.data.fill_(5.0)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        raw = self.sigmoid(self.fc2(x)).squeeze(-1)
        # Remap from (0,1) → (alpha_min, 1) so denominator stays positive
        return self.alpha_min + (1.0 - self.alpha_min) * raw


def _interval_size(scores, alpha, eps=1e-8):
    """
    Conformal interval width = 2 * Σscores / (α̃·(n+1) − 1).
    Denominator is clamped to be strictly positive.
    """
    n = len(scores)
    denom = alpha * (n + 1) - 1
    denom = torch.clamp(denom, min=eps)   # ← KEY FIX: prevent negative/zero
    r = scores.sum() / denom
    return 2 * r


def _fmt_time(seconds):
    """Format seconds into human-readable string."""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}min"
    else:
        return f"{seconds/3600:.1f}hr"


# ═══════════════════════════════════════════════════════════
# Main function  (CORRECTED + PROGRESS TRACKING)
# ═══════════════════════════════════════════════════════════
def conformal_predictions_AlphaNet(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    """
    Learned-α Conformal Prediction via AlphaNet.
    - FIXED: α̃ clamped to valid range (no negative intervals)
    - FIXED: LOO scores precomputed (massive speedup)
    - ADDED: tqdm progress bars + ETA estimates
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # --- numpy-ify if pandas ---
    y_train_np = y_train.to_numpy().ravel() if hasattr(y_train, "to_numpy") else np.asarray(y_train).ravel()
    y_test_np  = y_test.to_numpy().ravel()  if hasattr(y_test,  "to_numpy") else np.asarray(y_test).ravel()
    X_train_np = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    X_test_np  = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)

    # =================================================================
    # 1.  Split training data → fit + calibration
    # =================================================================
    n_total = len(X_train_np)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = X_train_np[fit_idx],   y_train_np[fit_idx]
    X_calib, y_calib = X_train_np[calib_idx],  y_train_np[calib_idx]
    n_calib = len(X_calib)

    # Compute alpha_min: must satisfy α̃·(n+1) − 1 > 0  →  α̃ > 1/(n+1)
    alpha_min = 1.0 / (n_calib + 1) + 0.01   # small margin above threshold
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =================================================================
    # 2.  Train base regression model
    # =================================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**model_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    # =================================================================
    # 3.  PRECOMPUTE all LOO scores  (KEY SPEEDUP)
    # =================================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = tqdm(range(n_calib), desc="    LOO precompute", leave=False) if HAS_TQDM else range(n_calib)

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        scores_loo = np.abs(base_model.predict(loo_X).ravel() - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # Build feature matrix (sum of LOO scores per sample)
    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =================================================================
    # 4.  Train AlphaNet (multiple runs, keep best)
    # =================================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net   = None
    best_loss  = np.inf
    history    = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    # USE PRECOMPUTED scores instead of calling predict()
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =================================================================
    # 5.  Compute conformal intervals on the TEST set
    # =================================================================
    print(f"  [Step 5/9] Computing conformal intervals on test set...")

    scores_calib = np.abs(base_model.predict(X_calib).ravel() - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small (denom={denom:.4f}); fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = base_model.predict(X_test_np).ravel()
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =================================================================
    # 6.  Metrics
    # =================================================================
    y_true_test = y_test_np
    is_covered  = ((y_true_test >= ymin) & (y_true_test <= ymax)).astype(int)
    coverage    = is_covered.mean()
    avg_width   = np.mean(ymax - ymin)
    rmse        = np.sqrt(mean_squared_error(y_true_test, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =================================================================
    # 7.  Summary & values DataFrames
    # =================================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true_test)),
        "y_true":       y_true_test,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =================================================================
    # 8.  Plotting (SAVE ONLY — NO SHOW)
    # =================================================================
    print(f"  [Step 8/9] Generating plots...")
    idx = np.arange(len(y_true_test))

    # --- PI plot ---
    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx, y_true_test, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx, y_pred_test, color="red", linewidth=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal – {model_name}")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel(f"{TARGET_LABEL}")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    # --- Training curves ---
    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =================================================================
    # 9.  Excel Export
    # =================================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        sn = _unique_sheet(wb, "AlphaNet_Summary")
        ws = wb.create_sheet(sn, 0)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        sn = _unique_sheet(wb, "AlphaNet_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(Image(buf_pi), "A1")

        sn = _unique_sheet(wb, "AlphaNet_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(Image(buf_tc), "A1")

        sn = _unique_sheet(wb, "AlphaNet_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet Conformal complete in {_fmt_time(total_time)}")
    return summary_df

# **Tuned Parameters**

In [19]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/concrete-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with CatBoost**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

In [21]:
conformal_predictions_NEXCP(
    model_class=CatBoostRegressor,
    best_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full CatBoost: 10.8 s/point with 1 worker, 3.8 s/point with 2


  NexCP-Full CatBoost: 95 test points, 8.9 min (2 workers)



===== NEXCP RESULTS =====
NexCP Split - Coverage: 97.895% | Width: 15.526
NexCP Full - Coverage: 92.632% | Width: 10.673
NexCP Jackknife+ aB - Coverage: 93.684% | Width: 11.590
NexCP CV+ - Coverage: 95.789% | Width: 15.304


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.978947,15.525703,2.988048,15.525703,
1,NexCP Full,0.926316,10.672879,2.988048,10.672879,18 of 95 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.936842,11.589517,3.116091,11.589517,
3,NexCP CV+,0.957895,15.303864,3.138901,15.303864,


In [22]:
conformal_predictions_online_super(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage      Width      RMSE        CWC Note
0          Online Split  0.968421  14.801705  2.988048  14.801705     
1            Online CV+  0.947368  13.875444  2.988048  13.875444     
2  Online Jackknife+ aB  0.915789  11.234568  3.078708  11.234568     
3                 SAOCP  0.915789  11.667612  2.988048  11.667612     
4                SF-OGD  0.905263  10.390838  2.988048  10.390838     
5                  FACI  0.915789  11.165963  2.988048  11.165963     
6                   ACI  0.915789  12.303287  2.988048  12.303287     
7            MFCS Split  0.968421  14.801705  2.988048  14.801705     
8             MFCS Full  0.905263  10.012177  2.875980  10.012177     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.968421,14.801705,2.988048,14.801705,
1,Online CV+,0.947368,13.875444,2.988048,13.875444,
2,Online Jackknife+ aB,0.915789,11.234568,3.078708,11.234568,
3,SAOCP,0.915789,11.667612,2.988048,11.667612,
4,SF-OGD,0.905263,10.390838,2.988048,10.390838,
5,FACI,0.915789,11.165963,2.988048,11.165963,
6,ACI,0.915789,12.303287,2.988048,12.303287,
7,MFCS Split,0.968421,14.801705,2.988048,14.801705,
8,MFCS Full,0.905263,10.012177,2.875980,10.012177,


In [23]:
conformal_predictions_COP(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (CatBoost) =====
Coverage : 91.579%
Avg Width: 12.269
RMSE     : 2.988
Final q  : 5.907


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.915789,12.268948,2.988048,0.1,0.05,0.5


In [24]:
conformal_predictions_EXTREME(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
        0.800000     3.060416             12 7.150004 -0.305193     7.527304
        0.810000     3.578794             11 7.223922 -0.326704     7.761619
        0.820000     4.246697             11 5.730089 -0.198557     7.425653
        0.824561     4.436664             10 6.480873 -0.280373     7.807115

===== EXTREME CONFORMAL PREDICTION (CatBoost) =====
Coverage : 91.579%
Width    : 7.807
RMSE     : 4.522
q_extreme: 7.807
u        : 4.437
#Exceed. : 10


In [25]:
conformal_predictions_AlphaNet(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=95 | alpha_min=0.0204
  [Step 2/9] Training base CatBoost model...


  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 95 LOO score vectors...


    LOO precompute:   0%|          | 0/95 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 667 epochs


    Run 1/4:   0%|          | 0/667 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/667 | Loss: 24.8089 | Size: 4.9524 | α̃: 0.9928 | ⏱ 0.17s/epoch | ETA: 10.6min


... (45 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 660/667 | Loss: 19.9419 | Size: 9.7890 | α̃: 0.5076 | ⏱ 0.02s/epoch | ETA: 0s
  λ=20.0 | Run 4/4 | Epoch 666/667 | Loss: 19.9413 | Size: 9.8888 | α̃: 0.5026 | ⏱ 0.02s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 19.9413
  [Step 4/9] Training done in 55s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (CatBoost)
  Learned α̃     : 0.5025
  Half-width (r) : 4.9433
  Coverage       : 88.421%
  Avg Width      : 9.8865
  RMSE           : 3.3421
  Total time     : 55s
    ├─ Precompute: 0s
    └─ Training  : 55s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 55s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.884211,9.886501,3.342137,0.1,0.497465,0.230411,0.502535,20.0,55.23
